In [1]:
# Load standard libraries
import os
import json
import sys
import pandas as pd
import requests
from dotenv import load_dotenv
import time

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mtick
from matplotlib.dates import MonthLocator, AutoDateLocator, ConciseDateFormatter, MO
from matplotlib.ticker import FuncFormatter, PercentFormatter, StrMethodFormatter

import datetime

from IPython.display import HTML

# Load env variables
load_dotenv("API_KEYS.env")
blockworks_api_key = os.getenv("BLOCKWORKS_API_KEY")
coingecko_api_key = os.getenv("COINGECKO_API_KEY")
dune_api_key = os.getenv("DUNE_API_KEY")
messari_api_key = os.getenv("MESSARI_API_KEY")
fred_api_key = os.getenv("FRED_API_KEY")

# Import data collection and processing functions
sys.path.append('src')
from data import blockworks
from data import coingecko
from data import metadao
from data import hyperliquid
from data import defillama
from data import dune
from data import messari
from data import yahoofinance
from data import fred
from data import data_processing
from data import protocol_index as pi

# Import utils functions
sys.path.append('utils')
import helper_functions
today = datetime.datetime.today()
date = helper_functions.format_date_with_suffix(today)

# Load output directories
blockworks_output_dir = 'data/raw/blockworks'
coingecko_output_dir = 'data/raw/coingecko'
defillama_output_dir = 'data/raw/defillama'
dune_output_dir = 'data/raw/dune'
fred_output_dir = 'data/raw/fred'
messari_output_dir = 'data/raw/messari'
metadao_output_dir = 'data/raw/metadao'
metadao_processed_dir = 'data/processed/metadao'
hyperliquid_output_dir = 'data/raw/hyperliquid'
yahoofinance_output_dir = 'data/raw/yahoofinance'

**Optional: clear existing raw data.** The pull functions now skip any file whose newest data point is within 3 days, and fetch the rest in parallel. Skip the next cell to repull only stale or missing files (fast). Run it to force a full repull. You can also pass `max_stale_days=None` to any DefiLlama or Messari pull to force a repull without deleting. With `DEFILLAMA_API_KEY` in `API_KEYS.env` the DefiLlama pulls run on the Pro host (`pro-api.llama.fi`, 1,000 req/min) at 16 requests in flight instead of 6 on the free host; the payloads are identical.

In [2]:
# Clear all existing raw data, you will be prompted for yes/no
directory_path = 'data/'
helper_functions.delete_directories(directory_path)


Found 899 files to delete


KeyboardInterrupt: Interrupted by user

**Blockworks.** The Research API key has a monthly usage cap. When it is reached every request answers `429 Monthly usage limit reached` and the pull below stops after its first request, prints the reset estimate and reports the skipped pairs, so the cell takes seconds rather than retrying ~100 requests. Files whose newest point is within 3 days are not repulled (pass `max_stale_days=None` to force), and each metric is fetched in one comma-separated request for all stale projects. Blockworks' current Data API (`/query/timeseries/...`, header `X-Blockworks-API-Key`) carries the same chain and lending metrics but needs a paid key with the `blockchains_unified` and `projects_protocols_unified` permissions; the Research API key is not accepted there.

In [ ]:
# BLOCKWORKS
start_date = '2020-01-01'
end_date = datetime.datetime.today().strftime('%Y-%m-%d')

In [ ]:
# Pull Blockworks data using protocol index
# Categories are pulled from config/protocol_index.json

blockworks_categories = ['smart_contract_chains',
                         'borrow_lend_protocols']

# Build categories dict with Blockworks slugs from the centralized index
categories = pi.get_categories_dict(categories=blockworks_categories, source='blockworks')

metrics = ['rev-usd',
           'transaction-fee-avg-usd',
           'transaction-total',
           'interest-total-usd',
           'nii-total-usd',
           'revenue-total-usd',
           'borrow-total-usd',
           'deposit-total-usd'
          ]

# Call the function
for category, assets in categories.items():
    print(f'{category}:')
    print()
    blockworks.pull_asset_data(assets.keys(), metrics, start_date, end_date, blockworks_api_key, blockworks_output_dir)
    print()
    print('---------------------------')
print('Done')

smart_contract_chains:



Blockworks rev-usd:   0%|          | 0/1 [00:00<?, ?pull/s]

Blockworks interest-total-usd:   0%|          | 0/1 [00:00<?, ?pull/s]

  Error fetching interest-total-usd for ethereum: 404 Client Error: Not Found for url: https://api.blockworks.com/v1/metrics/interest-total-usd?project=ethereum&start_date=2020-01-01&end_date=2026-10-10
  Error fetching interest-total-usd for solana: 404 Client Error: Not Found for url: https://api.blockworks.com/v1/metrics/interest-total-usd?project=solana&start_date=2020-01-01&end_date=2026-10-10
  Error fetching interest-total-usd for bnb: 404 Client Error: Not Found for url: https://api.blockworks.com/v1/metrics/interest-total-usd?project=bnb&start_date=2020-01-01&end_date=2026-10-10
  Error fetching interest-total-usd for arbitrum: 404 Client Error: Not Found for url: https://api.blockworks.com/v1/metrics/interest-total-usd?project=arbitrum&start_date=2020-01-01&end_date=2026-10-10
  Error fetching interest-total-usd for tron: 404 Client Error: Not Found for url: https://api.blockworks.com/v1/metrics/interest-total-usd?project=tron&start_date=2020-01-01&end_date=2026-10-10
  Error

Blockworks nii-total-usd:   0%|          | 0/1 [00:00<?, ?pull/s]

...

  Error fetching borrow-total-usd for euler: 404 Client Error: Not Found for url: https://api.blockworks.com/v1/metrics/borrow-total-usd?project=euler&start_date=2020-01-01&end_date=2026-10-10


Blockworks deposit-total-usd:   0%|          | 0/1 [00:00<?, ?pull/s]

  Error fetching deposit-total-usd for euler: 404 Client Error: Not Found for url: https://api.blockworks.com/v1/metrics/deposit-total-usd?project=euler&start_date=2020-01-01&end_date=2026-10-10


Blockworks deposit-total-usd: 100%|██████████| 1/1 [00:08<00:00,  8.16s/pull]


All rev-usd data saved to data/raw/blockworks/rev-usd/
All transaction-fee-avg-usd data saved to data/raw/blockworks/transaction-fee-avg-usd/
All transaction-total data saved to data/raw/blockworks/transaction-total/
All interest-total-usd data saved to data/raw/blockworks/interest-total-usd/
All nii-total-usd data saved to data/raw/blockworks/nii-total-usd/
All revenue-total-usd data saved to data/raw/blockworks/revenue-total-usd/
All borrow-total-usd data saved to data/raw/blockworks/borrow-total-usd/
All deposit-total-usd data saved to data/raw/blockworks/deposit-total-usd/

Failed to pull (19):
  borrow-total-usd:
    - euler
  deposit-total-usd:
    - euler
  interest-total-usd:
    - euler
  nii-total-usd:
    - euler
  rev-usd:
    - aave
    - euler
    - kamino
    - morpho
    - spark
  transaction-fee-avg-usd:
    - aave
    - euler
    - kamino
    - morpho
    - spark
  transaction-total:
    - aave
    - euler
    - kamino
    - morpho
    - spark

----------------------

In [ ]:
# Pull a single metric for ALL projects that track it.
# Project coverage is discovered from the Blockworks metrics catalog (GET /v1/metrics?identifier=...),
# e.g. https://docs.blockworksresearch.com/api-reference/metrics/hyperliquid/dex-spot-volume-total-usd

metrics_for_all_projects = ['rev-usd',                    # REV (real economic value)
                            'dex-spot-volume-total-usd']  # spot DEX volume

for metric in metrics_for_all_projects:
    print(f'{metric}:')
    print()
    blockworks.pull_metric_data(metric, start_date, end_date, blockworks_api_key, blockworks_output_dir)
    print()
    print('---------------------------')
print('Done')

rev-usd:

Found 39 projects tracking 'rev-usd'
Blockworks rev-usd: 6 fresh file(s) skipped, 33 to pull


Blockworks rev-usd: 100%|██████████| 2/2 [00:09<00:00,  4.98s/pull]



All rev-usd data saved to data/raw/blockworks/rev-usd/

---------------------------
dex-spot-volume-total-usd:

Found 23 projects tracking 'dex-spot-volume-total-usd'


Blockworks dex-spot-volume-total-usd:   0%|          | 0/2 [00:00<?, ?pull/s]

  Error fetching dex-spot-volume-total-usd for megaeth: 500 Server Error: Internal Server Error for url: https://api.blockworks.com/v1/metrics/dex-spot-volume-total-usd?project=megaeth&start_date=2020-01-01&end_date=2026-10-10


Blockworks dex-spot-volume-total-usd: 100%|██████████| 2/2 [00:37<00:00, 18.76s/pull]


All dex-spot-volume-total-usd data saved to data/raw/blockworks/dex-spot-volume-total-usd/

Failed to pull (1):
  dex-spot-volume-total-usd:
    - megaeth

---------------------------
Done


In [ ]:
# Calculate borrow/lend efficiency ratios for the Blockworks borrow/lend protocols
# Computes (per protocol, per date): interest generated / total borrows, net interest income / total borrows, revenue / total borrows, interest generated / total deposits, net interest income / total deposits, revenue / total deposits, and total borrows / total deposits, then saves as a processed data file.

# Pull the Blockworks slugs for the borrow/lend protocols from the centralized index
borrow_lend_protocols = pi.get_blockworks_assets_dict(category='borrow_lend_protocols')

data_processing.calculate_borrow_lend_ratios(
    protocols=borrow_lend_protocols,
    blockworks_output_dir=blockworks_output_dir,
    output_filepath='data/processed/borrow-lend/borrow-lend-ratios.json'
)

Processing aave...
  Computed ratios for 2139 dates
Processing morpho...
  Computed ratios for 1010 dates
Processing kamino...
  Computed ratios for 1093 dates
Processing euler...
  Computed ratios for 778 dates
Processing spark...
  Computed ratios for 1291 dates

Borrow/lend ratios saved to: data/processed/borrow-lend/borrow-lend-ratios.json
Protocols processed: 5
Done


In [ ]:
# DEFILLAMA
window = 'total30d' # How long a time period to look back when aggregating protocols to categories. Can be 'total24h', 'total7d', 'total30d', 'total1y', or 'totalAllTime'; defaults to 'totalAllTime'

In [ ]:
# Test pull all protocols by metric function
metrics = {'dailyFees': 'fees',
          'dailyRevenue': 'revenue',
          'dailyHoldersRevenue': 'holders-revenue',
          'tvl': 'tvl'
}
defillama.pull_all_protocols_metric(metrics, defillama_output_dir)

DefiLlama all-protocol metrics: 100%|██████████| 4/4 [00:11<00:00,  2.94s/pull]


All fees data saved to data/raw/defillama/fees/
All revenue data saved to data/raw/defillama/revenue/
All holders-revenue data saved to data/raw/defillama/holders-revenue/
All tvl data saved to data/raw/defillama/tvl/


In [ ]:
# Test pulling fees and revenue data function
metrics = {'dailyFees': 'fees',
          'dailyRevenue': 'revenue',
          'dailyHoldersRevenue': 'holders-revenue',
          'dailyIncentives': 'incentives',   # Pro only (DEFILLAMA_API_KEY); daily token incentives, USD
          'tvl': 'tvl'
}


categories = {
    'perps_dexs':{'hyperliquid': 'hyperliquid'
             },
    'smart_contract_chains': {'ethereum': 'ethereum'
             },
    'borrow_lend_protocols' : {'kamino': 'kamino'
            },
    'derivatives': {'derive': 'derive'},
    'misc': {'M0': 'M0',
             'axiom': 'axiom',
             'angstrom': 'angstrom'}
}

for category, assets in categories.items():
    print(f'{category}:') 
    print()
    defillama.pull_metric_data_by_asset(metrics, assets, defillama_output_dir)
    print()
    print('---------------------------')
print('Done')

perps_dexs:



DefiLlama metric data: 100%|██████████| 5/5 [00:02<00:00,  2.01pull/s]



All fees data saved to data/raw/defillama/fees/
All revenue data saved to data/raw/defillama/revenue/
All holders-revenue data saved to data/raw/defillama/holders-revenue/
All incentives data saved to data/raw/defillama/incentives/
All tvl data saved to data/raw/defillama/tvl/

---------------------------
smart_contract_chains:



DefiLlama metric data: 100%|██████████| 5/5 [00:11<00:00,  2.32s/pull]



All fees data saved to data/raw/defillama/fees/
All revenue data saved to data/raw/defillama/revenue/
All holders-revenue data saved to data/raw/defillama/holders-revenue/
All incentives data saved to data/raw/defillama/incentives/
All tvl data saved to data/raw/defillama/tvl/

---------------------------
borrow_lend_protocols:



...


All fees data saved to data/raw/defillama/fees/
All revenue data saved to data/raw/defillama/revenue/
All holders-revenue data saved to data/raw/defillama/holders-revenue/
All incentives data saved to data/raw/defillama/incentives/
All tvl data saved to data/raw/defillama/tvl/

---------------------------
misc:



DefiLlama metric data:  67%|██████▋   | 10/15 [00:01<00:00, 10.95pull/s]

  request failed: HTTP 400 HTTPError: https://pro-api.llama.fi/***/api/summary/fees/angstrom?dataType=dailyHoldersRevenue


DefiLlama metric data: 100%|██████████| 15/15 [00:03<00:00,  4.35pull/s]


All fees data saved to data/raw/defillama/fees/
All revenue data saved to data/raw/defillama/revenue/
All holders-revenue data saved to data/raw/defillama/holders-revenue/
All incentives data saved to data/raw/defillama/incentives/
All tvl data saved to data/raw/defillama/tvl/

Failed to pull (1):
  holders-revenue:
    - angstrom

---------------------------
Done


In [ ]:
# Test function to establish categories by parent protocol
parent_categories = data_processing.establish_parent_protocols(
    defillama_output_dir=defillama_output_dir, window=window
)

Establishing parent protocol categories using fees data...
Step 1: Collecting all parent-child relationships and metrics from fees, revenue, and TVL data...
  Checking fees data...
  Checking revenue data...
  Checking holders-revenue data...
  Checking tvl data...
  Found 7255 unique parent protocols

Step 2: Analyzing fees data to determine parent categories...

Parent protocols saved to: data/processed/parent_protocols.json
Total parent protocols: 1642
Done


In [ ]:
# Test function to pull total fees by category
parent_protocols_path = 'data/processed/parent_protocols.json'
    
category_fees = data_processing.calculate_total_fees_by_category(
        parent_protocols_path=parent_protocols_path
    )

Loading parent protocols from data/processed/parent_protocols.json...

Found 81 categories

Top 10 categories by total fees:
  1. Super App: $849,567,119
  2. Stablecoin Issuer: $543,298,098
  3. Dexs: $195,688,277
  4. Launchpad: $154,488,317
  5. Chain: $117,703,341
  6. Prediction Market: $114,481,043
  7. Liquid Staking: $97,743,732
  8. Lending: $94,163,813
  9. Staking Pool: $70,614,382
  10. RWA: $68,995,235

Total fees by category saved to: data/processed/total_fees_by_category.json
Done


In [ ]:
# Test function to pull top n parent protocols by category
metrics = {
    'dailyFees': 'fees',
    'dailyRevenue': 'revenue',
    'dailyHoldersRevenue': 'holders-revenue',
    'tvl': 'tvl'
}

categories = ['Derivatives',
              'Dexs',
              'Launchpad',
              'Lending',
              'Liquid Staking',
              'Options',
              'Super App']

n = 10

results = data_processing.top_n_protocols_by_category_by_metric(
    metrics=metrics,
    n=n,
    categories=categories,
    defillama_output_dir=defillama_output_dir,
    window=window
)

Loading parent protocol categories from data/processed/parent_protocols.json...
Loaded 1642 parent protocol categories

Processing all metrics using established categories...
Processing fees...
Processing revenue...
Processing holders-revenue...
Processing tvl...

Final output saved to: data/processed/top_protocols_by_category_by_metric.json
Done


In [ ]:
# Pull time series data for top parent protocols by dailyFees

# Define which metrics to pull
metrics = {
    'dailyFees': 'fees',
    'dailyRevenue': 'revenue',
    'dailyHoldersRevenue': 'holders-revenue',
    'tvl': 'tvl'
}

data_processing.pull_top_parent_protocols_timeseries(metrics, defillama_output_dir)

Pulling 4 metric(s) for 55 parent protocols (top by dailyFees)
DefiLlama metric data: 6 fresh file(s) skipped, 214 to pull


DefiLlama metric data:   0%|          | 1/214 [00:00<01:05,  3.27pull/s]

  request failed: HTTP 400 HTTPError: https://pro-api.llama.fi/***/api/summary/fees/arcus?dataType=dailyHoldersRevenue


DefiLlama metric data:   5%|▌         | 11/214 [00:00<00:07, 25.49pull/s]

  request failed: HTTP 400 HTTPError: https://pro-api.llama.fi/***/api/summary/fees/antarctic?dataType=dailyHoldersRevenue


...

  request failed: HTTP 400 HTTPError: https://pro-api.llama.fi/***/api/summary/fees/venus-finance?dataType=dailyRevenue


DefiLlama metric data:  99%|█████████▉| 212/214 [00:23<00:00,  2.14pull/s]

  request failed: HTTP 400 HTTPError: https://pro-api.llama.fi/***/api/protocol/venus-finance


DefiLlama metric data: 100%|██████████| 214/214 [00:44<00:00,  4.76pull/s]



Missing data by metric:
  fees: 4 of 55 protocols missing
    - astherus
    - maker
    - panoptic
    - venus-finance
  revenue: 5 of 55 protocols missing
    - astherus
    - extended
    - maker
    - panoptic
    - venus-finance
  holders-revenue: 26 of 55 protocols missing
    - antarctic
    - arcus
    - astherus
    - binance-staked-eth
    - binance-staked-sol
    - callput
    - circle
    - coinbase-wrapped-staked-eth
    - decibel
    - dolomite
    - dream
    - extended
    - fables
    - fomo
    - forward-industries-sol
    - liquid-collective
    - maker
    - pacifica
    - panoptic
    - project-x
    - rocket-pool
    - sofa.org
    - stochastic-finance
    - tonstakers
    - txflow
    - venus-finance
  tvl: 5 of 55 protocols missing
    - astherus
    - maker
    - panoptic
    - ramses-exchange-hl
    - venus-finance


In [ ]:
# Load parameters for following functions
start_date = '2021-01-01'  # Start date for the data (format: 'YYYY-MM-DD')
parent_categories_path = 'data/processed/parent_protocols.json'
top_protocols_path = 'data/processed/top_protocols_by_category_by_metric.json'
defillama_output_dir = 'data/raw/defillama'

In [ ]:
# Test function to calculate gross margins by category
category_gross_margins = data_processing.category_gross_margin_comparison(
    start_date=start_date,
    parent_categories_path=parent_categories_path,
    top_protocols_path=top_protocols_path,
    defillama_output_dir=defillama_output_dir
)

print(category_gross_margins.tail())

Found 55 top parent protocols with fees data
Found categories: {'Lending', 'Options', 'Super App', 'Derivatives', 'Dexs', 'Liquid Staking'}

Processing category: Lending
  Found 10 parent protocols in Lending
  Processing parent: aave
    aave avg gross margin: 13.53%
  Processing parent: hyperlend
    hyperlend avg gross margin: 16.10%
  Processing parent: dolomite
    dolomite avg gross margin: 20.21%
  Processing parent: euler
    euler avg gross margin: 5.32%
  Processing parent: venus-finance


DefiLlama metric data:   0%|          | 0/1 [00:00<?, ?pull/s]

  request failed: HTTP 400 HTTPError: https://pro-api.llama.fi/***/api/summary/fees/venus-finance?dataType=dailyFees


DefiLlama metric data: 100%|██████████| 1/1 [00:00<00:00, 16.36pull/s]



All fees data saved to data/raw/defillama/fees/

Failed to pull (1):
  fees:
    - venus-finance
    Failed to pull fees data for venus-finance, skipping this parent
    Skipping venus-finance - incomplete data
  Processing parent: maple-finance
    maple-finance avg gross margin: 13.36%
  Processing parent: world-liberty-financial
    world-liberty-financial avg gross margin: 100.00%
  Processing parent: compound-finance
    compound-finance avg gross margin: 11.48%
  Processing parent: morpho
    morpho avg gross margin: 0.00%
  Processing parent: justlend
    justlend avg gross margin: 5.22%
  Found 9 parent fee columns and 9 parent revenue columns
  Category Lending processed successfully

Processing category: Options
  Found 5 parent protocols in Options
  Processing parent: panoptic


...

DefiLlama metric data: 100%|██████████| 1/1 [00:00<00:00, 19.72pull/s]




All fees data saved to data/raw/defillama/fees/

Failed to pull (1):
  fees:
    - astherus
    Failed to pull fees data for astherus, skipping this parent
    Skipping astherus - incomplete data
  Processing parent: decibel
    decibel avg gross margin: 100.00%
  Processing parent: edgex
    edgex avg gross margin: 74.55%
  Processing parent: arcus
    arcus avg gross margin: 91.56%
  Processing parent: txflow
    txflow avg gross margin: 100.00%
  Processing parent: extended


DefiLlama metric data: 100%|██████████| 1/1 [00:00<00:00, 20.01pull/s]

  request failed: HTTP 400 HTTPError: https://pro-api.llama.fi/***/api/summary/fees/extended?dataType=dailyRevenue

All revenue data saved to data/raw/defillama/revenue/

Failed to pull (1):
  revenue:
    - extended
    Failed to pull revenue data for extended, skipping this parent
    Skipping extended - incomplete data
  Found 8 parent fee columns and 8 parent revenue columns
  Category Derivatives processed successfully

Processing category: Dexs
  Found 10 parent protocols in Dexs
  Processing parent: project-x


    project-x avg gross margin: 14.30%
  Processing parent: uniswap
    uniswap avg gross margin: 0.92%
  Processing parent: ramses-exchange-hl
    ramses-exchange-hl avg gross margin: 84.66%
  Processing parent: thorchain-dex
    thorchain-dex avg gross margin: 5.34%
  Processing parent: up
    up avg gross margin: 69.27%
  Processing parent: aerodrome
    aerodrome avg gross margin: 87.33%
  Processing parent: fables
    fables avg gross margin: 0.00%
  Processing parent: pharaoh-exchange
    pharaoh-exchange avg gross margin: 98.90%
  Processing parent: orca
    orca avg gross margin: 11.51%
  Processing parent: chainflip
    chainflip avg gross margin: 22.62%
  Found 10 parent fee columns and 10 parent revenue columns
  Category Dexs processed successfully

Processing category: Liquid Staking
  Found 10 parent protocols in Liquid Staking
  Processing parent: rocket-pool
    rocket-pool avg gross margin: 0.00%
  Processing parent: tonstakers
    tonstakers avg gross margin: 16.00%
 

In [ ]:
# Test function to calculate fees to tvl comparison
category_fees_vs_tvl = data_processing.category_fees_to_tvl_comparison(
    start_date=start_date,
    parent_categories_path=parent_categories_path,
    top_protocols_path=top_protocols_path,
    defillama_output_dir=defillama_output_dir
)

print(category_fees_vs_tvl.tail())

# Test function to calculate revenue to tvl comparison
category_revenue_vs_tvl = data_processing.category_revenue_to_tvl_comparison(
    start_date=start_date,
    parent_categories_path=parent_categories_path,
    top_protocols_path=top_protocols_path,
    defillama_output_dir=defillama_output_dir
)

print(category_revenue_vs_tvl.tail())

Found 55 top parent protocols with fees data
Found categories: {'Lending', 'Options', 'Super App', 'Derivatives', 'Dexs', 'Liquid Staking'}

Processing category: Lending
  Found 10 parent protocols in Lending
  Processing parent: aave
    aave avg fees/TVL: 1.20%
  Processing parent: hyperlend
    hyperlend avg fees/TVL: 1.15%
  Processing parent: dolomite
    dolomite avg fees/TVL: 1.73%
  Processing parent: euler
    euler avg fees/TVL: 1.87%
  Processing parent: venus-finance


DefiLlama metric data: 100%|██████████| 1/1 [00:00<00:00, 21.68pull/s]


  request failed: HTTP 400 HTTPError: https://pro-api.llama.fi/***/api/summary/fees/venus-finance?dataType=dailyFees

All fees data saved to data/raw/defillama/fees/

Failed to pull (1):
  fees:
    - venus-finance
    Failed to pull fees data for venus-finance


DefiLlama metric data: 100%|██████████| 1/1 [00:00<00:00, 41.14pull/s]

  request failed: HTTP 400 HTTPError: https://pro-api.llama.fi/***/api/protocol/venus-finance

All tvl data saved to data/raw/defillama/tvl/

Failed to pull (1):
  tvl:
    - venus-finance
    Failed to pull TVL data for venus-finance
    No fees data for parent venus-finance, skipping
  Processing parent: maple-finance
    maple-finance avg fees/TVL: 1.98%
  Processing parent: world-liberty-financial


...


All tvl data saved to data/raw/defillama/tvl/
    stakee avg revenue/TVL: 1.13%
  Processing parent: binance-staked-eth
    binance-staked-eth avg revenue/TVL: 0.29%
  Processing parent: stake.link


DefiLlama metric data: 100%|██████████| 1/1 [00:00<00:00,  2.12pull/s]



All revenue data saved to data/raw/defillama/revenue/


DefiLlama metric data: 100%|██████████| 1/1 [00:00<00:00,  1.10pull/s]



All tvl data saved to data/raw/defillama/tvl/
    stake.link avg revenue/TVL: 0.46%
  Found 10 parent revenue columns and 10 parent TVL columns
  Category Liquid Staking processed successfully

Revenue to TVL ratios by parent category saved to: data/processed/financial-metrics\revenue-to-tvl-by-category.json

Category-level aggregate revenue/TVL ratios (first 5 dates):

2021-01-01T00:00:00:
  Lending: 0.0016
  Options: nan
  Super App: nan
  Derivatives: nan
  Dexs: 0.0000
  Liquid Staking: nan

2021-01-01T23:00:00:
  Lending: 0.0016
  Options: nan
  Super App: nan
  Derivatives: nan
  Dexs: 0.0000
  Liquid Staking: 0.0000

2021-01-02T00:00:00:
  Lending: 0.0017
  Options: nan
  Super App: nan
  Derivatives: nan
  Dexs: 0.0000
  Liquid Staking: 0.0000

2021-01-02T23:00:00:
  Lending: 0.0017
  Options: nan
  Super App: nan
  Derivatives: nan
  Dexs: 0.0000
  Liquid Staking: 0.0000

2021-01-03T00:00:00:
  Lending: 0.0018
  Options: nan
  Super App: nan
  Derivatives: nan
  Dexs: 0.0000


In [ ]:
# Test pulling OI data for all dexs function

defillama.pull_all_open_interest(defillama_output_dir)

# Forward-fill missing values and save back to file
output_path = f"{defillama_output_dir}/open-interest/all-dexs.json"
with open(output_path, 'r') as f:
    data = json.load(f)

data = helper_functions.ffill_chart_breakdown(data)

with open(output_path, 'w') as f:
    json.dump(data, f, indent=2)


All open-interest data saved to data/raw/defillama/open-interest/


In [ ]:
# Test pulling all stablecoins function

defillama.pull_all_stablecoins(defillama_output_dir)


All stablecoins data saved to data/raw/defillama/stablecoins/


In [ ]:
# Test pulling specific stablecoins function
assets = ['USDC', 'USDT', 'USDe', 'BUSD', 'PYUSD']

defillama.pull_specific_stablecoins(assets, defillama_output_dir)

DefiLlama stablecoins: 100%|██████████| 5/5 [00:11<00:00,  2.32s/pull]


All stablecoins data saved to data/raw/defillama/stablecoins/


In [ ]:
# Test pulling stablecoin supply by network
networks = ['solana']

defillama.stablecoin_supply_by_network(networks, defillama_output_dir)

DefiLlama stablecoin supply by network: 100%|██████████| 1/1 [00:00<00:00,  1.22pull/s]


All stablecoins data saved to data/raw/defillama/stablecoins/


In [ ]:
# Test pulling top stablecoins by peg mechanism function
n = 20
pegMechanism = 'crypto-backed'
excluded_assets = ['Dai']

defillama.top_n_stablecoins_by_mcap(n, pegMechanism, excluded_assets, defillama_output_dir)

Data saved to data/raw/defillama/stablecoins/top-crypto-backed-stablecoins-by-mcap.json
Done


(['Sky Dollar',
  'Ethena USDe',
  'Falcon USD',
  'USDD',
  'United Stables',
  'GHO',
  'Polymarket USD',
  'Multipli rwaUSDi',
  'USD.AI',
  'apxUSD',
  'Re Protocol reUSD',
  'crvUSD',
  'Binance Peg BUSD',
  'Solstice USX',
  'flexUSD',
  'Mustang Finance',
  'Avalon USDa',
  'River Stablecoin',
  'Astherus',
  'Dola'],
 {'USDS': 'Sky Dollar',
  'USDe': 'Ethena USDe',
  'USDf': 'Falcon USD',
  'USDD': 'USDD',
  'U': 'United Stables',
  'GHO': 'GHO',
  'pUSD': 'Polymarket USD',
  'rwaUSDi': 'Multipli rwaUSDi',
  'USDai': 'USD.AI',
  'apxUSD': 'apxUSD',
  'reUSD': 'Re Protocol reUSD',
  'crvUSD': 'crvUSD',
  'BUSD': 'Binance Peg BUSD',
  'USX': 'Solstice USX',
  'FLEXUSD': 'flexUSD',
  'MUST': 'Mustang Finance',
  'USDA': 'Avalon USDa',
  'satUSD': 'River Stablecoin',
  'USDF': 'Astherus',
  'DOLA': 'Dola'})

In [ ]:
# DUNE

In [ ]:
# Test function to pull Dune data
query_ids = {'4078266': 'weekly-perps-dex-volumes',
             '4552208': 'maple-xirr',
             '5597706': 'hyperliquid-daily-volume-by-coin',
             '4307695': 'hyperliquid-usdc-supply',
             '3700662': 'sky-collateral'}

dune.fetch_and_save_dune_data(dune_api_key, query_ids, dune_output_dir)

Dune queries:  60%|██████    | 3/5 [00:01<00:00,  2.14pull/s]

  request failed: HTTP 402 HTTPError: https://api.dune.com/api/v1/query/4552208/results
  request failed: HTTP 404 HTTPError: https://api.dune.com/api/v1/query/5597706/results


Dune queries: 100%|██████████| 5/5 [00:02<00:00,  1.97pull/s]


All data saved to data/raw/dune/

Failed to pull (2):
  - hyperliquid-daily-volume-by-coin
  - maple-xirr


In [ ]:
# UPDATE PROTOCOL INDEX

In [ ]:
# --- Keep the protocol index up to date -------------------------------------
# Register the protocols pulled by top_n_protocols_by_category_by_metric (the
# `results` dict above) in config/protocol_index.json, then auto-fill their
# Messari / Blockworks slugs. Only protocols that came out of that DefiLlama
# pull are added. Nothing else from the DefiLlama universe.
#
# Non-destructive: protocols already in the index are skipped, and slugs already
# set are left untouched. You can hand-edit config/protocol_index.json to fix
# anything the functions get wrong, and your edits will not be overwritten.
#
# Display names come from the DefiLlama parent protocol ("Uniswap", not
# "Uniswap V3"), via the `name` field establish_parent_protocols records in
# data/processed/parent_protocols.json. repair_parent_display_names reports
# any entry in the synced category still carrying a sub-protocol's name; it is
# run as a dry run here, so nothing is rewritten. To apply the reported fixes,
# run it once with dry_run=False (it only touches names that are exactly one of
# the parent's own sub-protocol names, never a hand-chosen name).

# Map each DefiLlama category label (the keys inside `results`) -> the
# standardized category key used in protocol_index.json. Add rows to sync more.
sync_categories = {'Dexs': 'dexs',
                   'Launchpad': 'launchpads',
                   'Options': 'options',
                   'Super App': 'super_app'}

for dl_category, std_category in sync_categories.items():
    # Merge the {slug: display_name} maps across all pulled metrics for this category
    protocols = {}
    for metric_block in results.values():
        protocols.update(metric_block.get(dl_category, {}))

    print(f'{std_category} ({len(protocols)} protocols):')
    pi.add_defillama_protocols(protocols, category=std_category)
    pi.repair_parent_display_names(category=std_category, dry_run=True)
    pi.populate_remaining_slugs(
        standard_names=list(protocols),
        sources=['messari', 'blockworks'],
        blockworks_api_key=blockworks_api_key,
    )
    print('---------------------------')
print('Protocol index synced')

dexs (18 protocols):
add_defillama_protocols: added 0, skipped 18
  skipped: ['uniswap', 'aerodrome', 'orca', 'fables', 'thorchain-dex', 'pharaoh-exchange', 'ramses-exchange-hl', 'project-x', 'chainflip', 'up', 'nest', 'hydrex', 'velodrome', 'vvs-finance', 'quickswap', 'pulsex', 'balancer', 'ring-protocol'] (already in index)
repair_parent_display_names (dry run): would change 0, kept 26
  Messari all-assets file not found (c:\Users\JamesEvans\OneDrive - Reciprocal Ventures\Documents\Research\dd-tools\data\raw\messari\all-assets.json); run messari.pull_all_assets first. Skipping Messari.
populate_remaining_slugs: processed 11 protocol(s); filled 0 messari_slug, 0 blockworks_slug
---------------------------
super_app (20 protocols):
add_defillama_protocols: added 14, skipped 6
  added:   ['circle', 'pump', 'raydium', 'axiom', 'meteora', 'fomo', 'maker', 'phantom', 'pancakeswap', 'chainlink', 'bonkfun', 'near-protocol', 'arbitrum-foundation', 'lista-dao']
  skipped: ['hyperliquid', 'jupi

In [ ]:
# FRED

In [ ]:
# Test function to pull US M2 supply
series_id = 'M2SL'

fred.pull_m2_money_supply( series_id, fred_api_key, fred_output_dir)


All data saved to data/raw/fred/


In [ ]:
# MESSARI
start_date = '2021-01-01'
end_date = datetime.datetime.today().strftime('%Y-%m-%d')

In [ ]:
messari.pull_all_assets(messari_api_key, messari_output_dir)


All data saved to data/raw/messari/


In [ ]:
# Test function to pull metrics by assets
assets = {'bitcoin': 'bitcoin',
         }

metrics = ['futures-open-interest',
           'futures-funding-rate',
           'futures-volume',
           'marketcap',
           'price',
           'supply'
          ]


messari.pull_asset_data(assets, metrics, start_date, end_date, messari_api_key, messari_output_dir)

Messari asset data: 100%|██████████| 6/6 [00:01<00:00,  4.19pull/s]


All futures-open-interest data saved to data/raw/messari/futures-open-interest/
All futures-funding-rate data saved to data/raw/messari/futures-funding-rate/
All futures-volume data saved to data/raw/messari/futures-volume/
All marketcap data saved to data/raw/messari/marketcap/
All price data saved to data/raw/messari/price/
All supply data saved to data/raw/messari/supply/


In [ ]:
# Test function to save down all networks tracked by Messari's API then save save to a dictionary to be used in the pull_network_data function

metric_category = 'ecosystem'
metric_extract = 'ecosystemCommits24Hour'
excluded_slugs = []
n = 20
extract_dict = messari.fetch_and_save_top_n_networks(metric_category, metric_extract, excluded_slugs, n, messari_api_key, messari_output_dir)

In [ ]:
# Function to pull data for a specific network
networks = extract_dict.keys()
print(networks)

metrics = [metric_category]


messari.pull_network_data(networks, metrics, start_date, end_date, messari_api_key, messari_output_dir)

dict_keys(['ethereum', 'polygon-pos', 'internet-computer', 'polkadot', 'kusama', 'stellar', 'avalanche', 'celestia', 'binance-smart-chain', 'base', 'solana', 'optimism', 'cardano', 'tron', 'near', 'bitcoin', 'celo', 'arbitrum-one', 'starknet', 'algorand'])


Messari network data: 100%|██████████| 20/20 [00:05<00:00,  3.72pull/s]


All ecosystem data saved to data/raw/messari/ecosystem/


In [ ]:
# Test function to pull data for individual stablecoins
stablecoins = ['tether', 'circle-usdc-stablecoin']
metrics = ['supply']
messari.pull_stablecoin_data(stablecoins, metrics, start_date, end_date, messari_api_key, messari_output_dir)

Messari stablecoin data: 100%|██████████| 2/2 [00:02<00:00,  1.12s/pull]


All supply data saved to data/raw/messari/supply/


In [ ]:
# Test function to pull data for individual exchanges

exchanges = ['coinbase', 'gemini', 'kraken']
metrics = ['volume', 'futures-volume']
messari.pull_exchange_data(exchanges, metrics, start_date, end_date, messari_api_key, messari_output_dir)

Messari exchange data: 100%|██████████| 3/3 [00:03<00:00,  1.29s/pull]


All volume data saved to data/raw/messari/volume/
All futures-volume data saved to data/raw/messari/futures-volume/


In [ ]:
# COINGECKO
category = 'metadao-launchpad'  # https://www.coingecko.com/en/categories/metadao-launchpad
days = 365  # demo API tier max; daily granularity is automatic for days > 90

In [ ]:
# Pull daily USD price history for every token in a CoinGecko category.
# The token list is discovered dynamically from the category, so newly
# launched projects are picked up on each run.

assets = coingecko.get_category_coins(category, coingecko_api_key, coingecko_output_dir)
print(f'{len(assets)} tokens: {list(assets)}')
print()

coingecko.pull_asset_data(assets, coingecko_api_key, coingecko_output_dir, days=days)
print('Done')

12 tokens: ['solomon', 'p2p-protocol', 'umbra', 'rip-cars', 'avici', 'omnipair', 'laso-finance', 'loyal', 'jurassic-finance', 'paystream', 'zklsol', 'ranger']



CoinGecko price data: 100%|██████████| 12/12 [00:26<00:00,  2.24s/pull]


All price data saved to data/raw/coingecko/price/
Done


In [ ]:
# METADAO
# MetaDAO launchpad raises: ICO price from the on-chain launch accounts, first-week
# prices from MetaDAO's market API (no CoinGecko). Every participant in a MetaDAO sale
# pays the same price: the program mints a fixed 10M tokens to funders as a group, so
# ico_price_usd = usdc_accepted / 10,000,000. The steps below fill config/metadao_launches.json
# (hand-editable; the pull only adds what is missing) and build the hourly VWAP file
# that example-analysis.ipynb charts.
launch_config = 'config/metadao_launches.json'
n_days = 7              # first n days of trading to keep (hourly)
min_accepted_usd = 100_000   # new launches below this are written with include=false


In [ ]:
# Refresh: run this any time to check MetaDAO for new launches. Launches already on file are
# never repulled (a completed raise does not change). One fresh tickers call decides whether a
# new DAO launched since the last pull; only then are the launch accounts repulled and the new
# rows added to the config (existing rows untouched). Pairs and swaps are fetched only for
# launches that lack them or whose first n_days had not finished last time; the daily market
# data (tie-out only) is repulled only when new swaps came in; the processed hourly file is
# rebuilt only when an input changed. One line per step. Manual override to repull every
# launch account: metadao.pull_launches(metadao_output_dir, force=True).
metadao.refresh(launch_config, metadao_output_dir, metadao_processed_dir, n_days=n_days, min_accepted_usd=min_accepted_usd)


launch accounts: repulled v0.4 (6), v0.5 (7), v0.6 (41), v0.7 (153), v0.8 (0) in 2s
config: 69 launches, 16 included; no new launches
pairs: 14/16 included launches resolved; new: UMBRA, AVICI, LOYAL, ZKFG, PAYS, SOLO, RNGR, P2P, RAWR, LASO, CRED, CARS, ORDR, NECTAR; closed before the swap feed starts (2025-10-09), no price history: MTN, OMFG


MetaDAO launch swaps: 100%|██████████| 14/14 [02:15<00:00,  9.68s/pull]


swaps: 14 launches complete on file; pulled UMBRA (22,794 swaps), AVICI (22,445 swaps), LOYAL (7,215 swaps), ZKFG (7,090 swaps), PAYS (6,656 swaps), SOLO (17,443 swaps), RNGR (8,160 swaps), P2P (4,562 swaps), RAWR (6,884 swaps), LASO (2,509 swaps), CRED (17,998 swaps), CARS (3,684 swaps), ORDR (1,878 swaps), NECTAR (5,077 swaps); no pair: MTN, OMFG
[pull_market_data] failed: HTTP 400 HTTPError: 400 Client Error: Bad Request for url: https://market-api.metadao.fi/api/market-data?startDate=2025-10-01&endDate=2026-10-10
data/processed/metadao/launch-prices-7d.json: 14 launches x 168 hourly points; no swap history: MTN, OMFG; tie-out clean (ICO price identity on every launch; daily VWAP within 0.5% of MetaDAO on all 0 launch-days)


{'_metadata': {'n_days': 7,
  'freq': '1h',
  'price': 'hourly VWAP = USDC volume / token volume of every swap on the launch pair (MetaDAO spot AMM)',
  'multiple': 'vwap / ico_price_usd; ico_price_usd = usdc_accepted / 10,000,000 tokens',
  'built': '2026-10-10T14:11:53Z'},
 'launches': {'PRVT6TB7uss3FrUd2D9xs2zqDBsa3GbMJMwCQsgmeta': {'ticker': 'UMBRA',
   'name': 'Umbra',
   'ico_price_usd': 0.3,
   'usdc_accepted': 3000000.0,
   'usdc_committed': 154943746.0,
   'sale_closed': '2025-10-10',
   'trading_start': '2025-10-10T15:02:58Z',
   'pair_id': 'BLkBSE96kQys7SrMioKxeMiVbeo4Ckk2Y4n1JphKxYnv',
   'hours': [{'hour': 0,
     'ts': '2025-10-10T15:02:58Z',
     'vwap': 0.9813516648894771,
     'usdc_volume': 2156238.1975610005,
     'token_volume': 2197212.553569003,
     'trades': 1052,
     'multiple': 3.271172216298257},
    {'hour': 1,
     'ts': '2025-10-10T16:02:58Z',
     'vwap': 1.354299644356585,
     'usdc_volume': 1690822.0285939996,
     'token_volume': 1248484.436690001,
 

In [ ]:
# The individual steps refresh() runs, for debugging or partial reruns.
# 1. Launch accounts -> config. getProgramAccounts on each launchpad program (v0.4-v0.8), saved
# as returned under launches/ (existing files are kept; force=True repulls all, a few seconds),
# decoded, and added to the config where missing.
metadao.pull_launches(metadao_output_dir)
decoded = metadao.decode_launches(metadao_output_dir)
metadao.fill_launch_config(launch_config, decoded, metadao_output_dir, min_accepted_usd=min_accepted_usd)


launch accounts: 5 program file(s) fresh (oldest 3m ago), kept
decoded 206 launch accounts; 1 skipped as undecodable: v0.4 7uDTCorL... (v0.4 layout leaves 71 non-zero trailing bytes)
config: 69 launches, 16 included; no new launches


{'_metadata': {'description': 'MetaDAO launchpad raises. Hand-editable: metadao.fill_launch_config only adds missing launches and missing keys, it never overwrites a value. Set include=false to drop a launch from the charts without deleting it.',
  'price_formula': 'ico_price_usd = usdc_accepted / tokens_to_participants (the program mints a fixed 10,000,000 tokens to funders as a group, pro rata to accepted USDC)',
  'usdc_accepted': 'total_approved_amount (v0.7+), final_raise_amount (v0.6) or committed (v0.4/v0.5) from the on-chain Launch account, in USDC',
  'programs': {'v0.4': 'AfJJJ5UqxhBKoE3grkKAZZsoXDE9kncbMKvqSHGsCNrE',
   'v0.5': 'mooNhciQJi1LqHDmse2JPic2NqG2PXCanbE3ZYzP3qA',
   'v0.6': 'MooNyh4CBUYEKyXVnjGYQ8mEiJDpGvJMdvrZx1iGeHV',
   'v0.7': 'moontUzsdepotRGe5xsfip7vLPTJnVuafqdUWexVnPM',
   'v0.8': 'moonDJUoHteKkGATejA5bdJVwJ6V6Dg74gyqyJTx73n'},
  'launch_discriminator_base64': 'kDMzo85V1SY='},
 'launches': {'9BEzsxayF8rx4AEsN9duFBn3NQDUf1hX3eTWPJQmeta': {'ticker': 'KOL',
  

In [ ]:
# 2. AMM pair per included launch (pool id + exact trading start), filled into the config, then
# every swap on that pair for its first n_days (raw, swaps/{mint}.json). Both only touch launches
# that still lack a pair / a completed window; launches that closed before the swap feed starts
# (2025-10-09: Omnipair, mtnCapital) are reported and skipped.
metadao.pull_pairs(launch_config, metadao_output_dir)
metadao.pull_launch_swaps(launch_config, metadao_output_dir, n_days=n_days)


pairs: 14/16 included launches resolved; closed before the swap feed starts (2025-10-09), no price history: MTN, OMFG
swaps: 14 launches complete on file; no pair: MTN, OMFG


0

In [ ]:
# 3. Daily market data (tie-out only; kept once on file, force=True repulls), then the processed
# hourly VWAP file the chart reads (rebuilt only when an input changed; force=True rebuilds).
# The build line reports the ICO-price identity check and the swap-derived daily VWAP vs
# MetaDAO's daily endpoint on every full launch-day; any day off by more than 0.5% is listed.
metadao.pull_market_data(metadao_output_dir)
metadao.build_launch_prices(launch_config, metadao_output_dir, metadao_processed_dir, n_days=n_days);


[pull_market_data] failed: HTTP 400 HTTPError: 400 Client Error: Bad Request for url: https://market-api.metadao.fi/api/market-data?startDate=2025-10-01&endDate=2026-10-10
data/processed/metadao/launch-prices-7d.json: up to date (14 launches, built 0m ago), kept


In [ ]:
# HYPERLIQUID
# AQAv2 reserve yield: each UTC date the node charges the treasury deployer's native-USDC balance on
# HyperEVM (90% of the USDC backing HyperCore, the docs' 9:1 split) at the cost-adjusted AQA rate / 365.25,
# sums 30 days from activation (2026-08-27) and pays the Assistance Fund 8 days later. Official daily
# series from the /info endpoint (aqav2InterestHistory, no key); payments from the system interest
# address's ledger; the treasury balance is re-read at each 00:00 UTC block from an archive RPC as an
# independent tie-out (rpc.hyperliquid.xyz/evm is not an archive node); EFFR from FRED is the reference
# rate the forward estimate in example-analysis.ipynb is linked to.
token = 0             # AQAv2 token index (0 = USDC, the only aligned asset today)
max_stale_days = 0    # repull the series when its newest date is before today UTC

In [ ]:
# Refresh: the three pulls (each skipped when fresh; the treasury balances are append-only), then the
# period table with the tie-outs printed on one line: the interest identity on every day, the period
# boundaries, the treasury balance vs the charged base, and paid vs accrued once payments land.
periods = hyperliquid.refresh(hyperliquid_output_dir, token=token, max_stale_days=max_stale_days)
fred.pull_series(hyperliquid.DEFAULT_REFERENCE_SERIES, fred_api_key, fred_output_dir, observation_start='2026-01-01')
periods

aqav2 interest history (USDC): pulled 56 rows, 2026-08-16..2026-10-10 (45 active)
aqav2 payments (USDC): 6 ledger update(s), 2 payment(s) to the Assistance Fund; period 2 due 2026-11-02


treasury balances: 100%|██████████| 56/56 [03:36<00:00,  3.86s/pull]


treasury balances (USDC): read 56 new date(s) from https://rpc.hyperlend.finance/archive (56 on file, 2026-08-16..2026-10-10)
aqav2 (USDC): 45 active days in 2 period(s), 1 paid; identity holds to 9.9e-09 on every day; base = treasury balance at midnight on all 56 days read (52 exact to the cent); tie-out clean
FRED EFFR: pulled 195 observations, 2026-01-02..2026-10-08


,start,end,payment_date,days_sampled,status,interest_accrued,avg_supply,avg_rate,actual_paid,paid_date,difference,payment_tx
period,,,,,,,,,,,,
1,2026-08-27,2026-09-25,2026-10-03,30,paid,1.458078e+07,5.649550e+09,0.031419,1.458078e+07,2026-10-03,0.0,0x00000000000000000000000000000000000000000000...
2,2026-09-26,2026-10-25,2026-11-02,15,in-progress,8.041252e+06,6.058227e+09,0.032323,NaN,NaT,NaN,NaN


In [ ]:
# YAHOO FINANCE
start_date = "2020-01-01"
end_date   = datetime.datetime.today().strftime("%Y-%m-%d")

In [ ]:
# Test pull mcap to annualized metric multiple
tickers   = ['COIN', 'CRCL', 'HOOD', 'GEMI', 'LVS', 'DKNG', 'GOOGL', 'AAPL', 'MSFT', 'CME', 'ICE', 'NDAQ', 'FNF', 'SCHW', 'IBKR']
metrics = {'Total Revenue': 'total-revenue',
           'EBITDA': 'ebitda'}

yahoofinance.pull_mcap_to_annualized_metric_multiple(tickers, metrics, start_date, end_date, yahoofinance_output_dir)

Yahoo Finance multiples: 100%|██████████| 15/15 [00:12<00:00,  1.18pull/s]


All mcap-to-annualized-total-revenue data saved to data/raw/yahoofinance/mcap-to-annualized-total-revenue/
All mcap-to-annualized-ebitda data saved to data/raw/yahoofinance/mcap-to-annualized-ebitda/


In [ ]:
# VALUATION MULTIPLES
start_date = '2021-01-01'
end_date = datetime.datetime.today().strftime('%Y-%m-%d')

In [ ]:
# Pull top metrics from Messari using protocol index
# Categories are pulled from config/protocol_index.json

valuation_categories = ['perps_dexs', 'smart_contract_chains', 'borrow_lend_protocols', 'launchpads', 'liquid_staking', 'options', 'physical_tcg', 'dexs']

# Build categories dict with Messari slugs from the centralized index
categories = pi.get_categories_dict(categories=valuation_categories, source='messari')

metrics = ['marketcap',
           'price'
          ]

# Call the function
for category, assets in categories.items():
    print(f'{category}:') 
    print()
    messari.pull_asset_data(assets, metrics, start_date, end_date, messari_api_key, messari_output_dir)
    print()
    print('---------------------------')
print('Done')

perps_dexs:



Messari asset data: 100%|██████████| 16/16 [00:01<00:00,  9.17pull/s]



All marketcap data saved to data/raw/messari/marketcap/
All price data saved to data/raw/messari/price/

---------------------------
smart_contract_chains:



Messari asset data:  83%|████████▎ | 10/12 [00:00<00:00, 15.89pull/s]

  request failed: HTTP 404 HTTPError: https://api.messari.io/metrics/v2/assets/base/metrics/marketcap/time-series/1d?start=2021-01-01T00%3A00%3A00Z&end=2026-10-10T00%3A00%3A00Z
  request failed: HTTP 404 HTTPError: https://api.messari.io/metrics/v2/assets/base/metrics/price/time-series/1d?start=2021-01-01T00%3A00%3A00Z&end=2026-10-10T00%3A00%3A00Z


...


All marketcap data saved to data/raw/messari/marketcap/
All price data saved to data/raw/messari/price/

---------------------------
physical_tcg:



Messari asset data: 100%|██████████| 2/2 [00:00<00:00, 10.75pull/s]



All marketcap data saved to data/raw/messari/marketcap/
All price data saved to data/raw/messari/price/

---------------------------
dexs:



Messari asset data: 100%|██████████| 14/14 [00:00<00:00, 14.91pull/s]


All marketcap data saved to data/raw/messari/marketcap/
All price data saved to data/raw/messari/price/

---------------------------
Done


In [ ]:
# Pull bottom metrics from DefiLlama using protocol index
# Categories are pulled from config/protocol_index.json

# Metrics can accept 'dailyFees': 'fees', 'dailyRevenue': 'revenue', 'dailyHoldersRevenue': 'holders-revenue',
# 'dailyIncentives': 'incentives' (Pro only) and 'tvl': 'tvl'. Earnings is derived in the next cell.
metrics = {'dailyFees': 'fees',
          'dailyRevenue': 'revenue',
          'dailyHoldersRevenue': 'holders-revenue',
          'dailyIncentives': 'incentives',   # Pro only (DEFILLAMA_API_KEY); daily token incentives, USD
          'tvl': 'tvl'
}

# Build categories dict with DefiLlama slugs from the centralized index
categories = pi.get_categories_dict(categories=valuation_categories, source='defillama')

for category, assets in categories.items():
    print(f'{category}:') 
    print()
    defillama.pull_metric_data_by_asset(metrics, assets, defillama_output_dir)
    print()
    print('---------------------------')
print('Done')

perps_dexs:

DefiLlama metric data: 17 fresh file(s) skipped, 23 to pull


DefiLlama metric data:  43%|████▎     | 10/23 [00:02<00:01,  7.29pull/s]

  request failed: HTTP 400 HTTPError: https://pro-api.llama.fi/***/api/summary/fees/aster?dataType=dailyRevenue


DefiLlama metric data: 100%|██████████| 23/23 [00:03<00:00,  7.05pull/s]



All fees data saved to data/raw/defillama/fees/
All revenue data saved to data/raw/defillama/revenue/
All holders-revenue data saved to data/raw/defillama/holders-revenue/
All incentives data saved to data/raw/defillama/incentives/
All tvl data saved to data/raw/defillama/tvl/

Failed to pull (1):
  revenue:
    - aster

---------------------------
smart_contract_chains:

DefiLlama metric data: 4 fresh file(s) skipped, 26 to pull


...

  request failed: HTTP 400 HTTPError: https://pro-api.llama.fi/***/api/protocol/ramses-exchange-hl


DefiLlama metric data:  88%|████████▊ | 79/90 [00:03<00:00, 23.21pull/s]

  request failed: HTTP 400 HTTPError: https://pro-api.llama.fi/***/api/summary/fees/fables?dataType=dailyHoldersRevenue


DefiLlama metric data: 100%|██████████| 90/90 [00:21<00:00,  4.16pull/s]


All fees data saved to data/raw/defillama/fees/
All revenue data saved to data/raw/defillama/revenue/
All holders-revenue data saved to data/raw/defillama/holders-revenue/
All incentives data saved to data/raw/defillama/incentives/
All tvl data saved to data/raw/defillama/tvl/

Failed to pull (8):
  fees:
    - dragon-swap
  holders-revenue:
    - dragon-swap
    - fables
    - project-x
    - ring-protocol
  revenue:
    - dragon-swap
  tvl:
    - dragon-swap
    - ramses-exchange-hl

---------------------------
Done


In [ ]:
# Derive daily earnings = revenue - incentives for every protocol pulled above (DefiLlama's Earnings
# definition). Written to data/raw/defillama/earnings/{Name}.json in the same shape as fees / revenue,
# so every chart and multiple that takes a metric dir can use 'earnings'. A protocol with no incentives
# tracked on DefiLlama gets earnings = revenue (listed below). Needs the incentives files (Pro key).
for category, assets in categories.items():
    print(f'{category}:')
    data_processing.build_earnings_series(assets, defillama_output_dir)

# Tie-out: monthly sums of each incentives file vs the Incentives line of DefiLlama's financial
# statement. Anything other than 'match' is a data gap worth knowing about before using earnings.
all_assets = {slug: name for assets in categories.values() for slug, name in assets.items()}
tieout = data_processing.check_incentives_tieout(all_assets, defillama_output_dir, months=12)
if not tieout.empty:
    display(tieout[tieout.status != 'match'])


perps_dexs:


DefiLlama metric data: 100%|██████████| 1/1 [00:00<00:00, 68.88pull/s]


  request failed: HTTP 400 HTTPError: https://pro-api.llama.fi/***/api/summary/fees/aster?dataType=dailyRevenue
  earnings: no revenue file for aster, skipped
  earnings = revenue (no incentives tracked on DefiLlama): hyperliquid, lighter, drift, dydx, jupiter, apex-protocol
smart_contract_chains:
  earnings = revenue (no incentives tracked on DefiLlama): solana, bnb, arbitrum, tron, base
borrow_lend_protocols:
  earnings = revenue (no incentives tracked on DefiLlama): kamino, maple-finance, spark
launchpads:
liquid_staking:
  earnings = revenue (no incentives tracked on DefiLlama): jito, rocket-pool, sanctum, marinade, kinetiq
options:
  earnings = revenue (no incentives tracked on DefiLlama): derive
physical_tcg:
  earnings = revenue (no incentives tracked on DefiLlama): collector-crypt, courtyard, beezie, phygitals, monster, pull.fun, dyli, shiny, yourgrails, chapool, gemint, pumpbox
dexs:


DefiLlama metric data: 100%|██████████| 1/1 [00:00<00:00, 21.37pull/s]

  request failed: HTTP 400 HTTPError: https://pro-api.llama.fi/***/api/summary/fees/dragon-swap?dataType=dailyRevenue
  earnings: no revenue file for dragon-swap, skipped


  earnings = revenue (no incentives tracked on DefiLlama): chainflip, project-x, ring-protocol, pulsex, metadao, ramses-exchange-hl, paycash, vvs-finance, aquarius-stellar, deepstate, humidifi, fables, hydrex
incentives tie-out: 229 month(s) match, 0 do not


,protocol,emission_slug,month,chart_usd,statement_usd,diff,status


In [ ]:
# Universe earnings: every protocol DefiLlama tracks, rolled up to parents, from the universe
# snapshots already pulled above (fees, revenue and holders-revenue overviews, the /protocols list for
# mcap) plus one Pro request for the incentives snapshot. Stores fees / revenue / holders revenue /
# incentives as reported for each of DefiLlama's trailing windows (1, 7, 30 and 365 days; it reports
# no other totals universe-wide); the EARNINGS RANKINGS cells in example-analysis.ipynb pick the
# window with data_processing.load_universe_earnings(window_days=...). Rules: data_processing.UNIVERSE_EARNINGS_RULES.
defillama.pull_emissions_breakdown(defillama_output_dir)
universe = data_processing.build_universe_earnings(defillama_output_dir)  # returns the 30d view
universe.sort_values('ann_earnings', ascending=False).head(10)[
    ['name', 'category', 'mcap', 'ann_fees', 'ann_revenue', 'ann_holders_rev', 'ann_earnings',
     'earnings_minus_holders_rev', 'earnings_yield', 'incentives_status']]

In [ ]:
# Process valuation multiples for protocols & categories (FDV/MCAP / fees/revenue/holders-revenue)
# Uses standardized names from protocol index

# Build categories dict with standardized names from the centralized index
categories = pi.get_standard_names_dict(categories=valuation_categories)

data_processing.process_asset_metrics_data(
    categories=categories,
    output_filepath="data/processed/valuation-multiples/all-protocols-by-category.json"
)


Determining global date range...
Global date range: 2015-08-07 to 2026-10-10 (4083 days)

Processing category: perps_dexs
  Processing asset: hyperliquid
  Processing asset: lighter
  Processing asset: aster
    Replaced 2 outlier multiple values for aster (>100x from trailing 90d median) across 2 multiples
  Processing asset: drift
    Replaced 158 outlier multiple values for drift (>100x from trailing 90d median) across 6 multiples
  Processing asset: dydx
    Replaced 12 outlier multiple values for dydx (>100x from trailing 90d median) across 6 multiples
  Processing asset: jupiter
  Processing asset: gmx
    Replaced 2 outlier multiple values for gmx (>100x from trailing 90d median) across 2 multiples
  Processing asset: apex-protocol
    Replaced 88 outlier multiple values for apex-protocol (>100x from trailing 90d median) across 5 multiples

Processing category: smart_contract_chains
  Processing asset: ethereum
    Error loading TVL for ethereum: 'date'
  Processing asset: sola

In [ ]:
# Test functions to pull aggregated valuation multiples by category

# Calculate weighted multiples
breakout_assets = {
    'perps_dexs': None,
    'smart_contract_chains': None,
    'borrow_lend_protocols': None,
    'launchpads': None,
    'liquid_staking': None,
    'depin': None,
    'yield': None,
    'nft_marketplace': None,
    'physical_tcg': None,
    'misc': None
}

data_processing.calculate_weighted_multiples(
    input_filepath="data/processed/valuation-multiples/all-protocols-by-category.json",
    categories=categories,
    breakout_assets=breakout_assets,
    start_date='2021-01-01',
    output_filepath="data/processed/valuation-multiples/weighted-multiples-by-category.json"
)

Weighted multiples data saved to data/processed/valuation-multiples/weighted-multiples-by-category.json


In [ ]:
# OTHER PROCESSED DATA

In [ ]:
# Test function to process and save annualized volatility data as JSON
trailing_window=7 # defaults to 7

data_processing.process_annualized_volatility_data(start_date, trailing_window)

Saved aave data to data/processed/annualized-volatility/aave.json
Saved aerodrome data to data/processed/annualized-volatility/aerodrome.json
Saved apex-protocol data to data/processed/annualized-volatility/apex-protocol.json
Saved arbitrum data to data/processed/annualized-volatility/arbitrum.json
Saved aster data to data/processed/annualized-volatility/aster.json
Saved bitcoin data to data/processed/annualized-volatility/bitcoin.json
Saved bnb data to data/processed/annualized-volatility/bnb.json
Saved chainflip data to data/processed/annualized-volatility/chainflip.json
Saved collector-crypt data to data/processed/annualized-volatility/collector-crypt.json
Saved compound-finance data to data/processed/annualized-volatility/compound-finance.json
Saved derive data to data/processed/annualized-volatility/derive.json
Saved drift data to data/processed/annualized-volatility/drift.json
Saved dydx data to data/processed/annualized-volatility/dydx.json
Saved ether-fi data to data/processed/